# 第17讲：整理连接诊断技能库

练习保护应急技能、原生状态管理、模型自主整理与恢复。所有技能和现场数据均为构造教学数据。题面、运行和停止方式见 [README](README.md)。请使用仓库固定依赖的 Python 内核，每轮重新启动内核；本 Notebook 不读取个人 Hermes Home。


In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd()
if not (HERE / 'curation_workshop.py').is_file():
    HERE = HERE / 'examples/17-curator'
assert (HERE / 'curation_workshop.py').is_file(), '请从本讲或仓库根目录打开'
sys.path.insert(0, str(HERE))
from curation_workshop import CurationLab, tree
lab = CurationLab()
lab.seed()
print('本轮目录：', lab.root)


## 1. 先观察状态，不请求模型

`seed()` 通过原生 `skill_manage` 创建四项技能，并构造活动时间、保护应急手册。下面直接调用原生自动转换；核对 active、stale、archived 与 pinned 的关系。


In [ ]:
from agent import curator, curator_backup
from tools import skill_usage

counts = curator.apply_automatic_transitions(now=lab.clock)
lab.observe_transitions(counts)
{n: skill_usage.get_record(n) for n in ('connection-pool-triage', 'connection-leak-investigation', 'disaster-recovery', 'retired-db-debug')}


## 2. 整理前先留下两项诊断基线
下面调用模型，容量与疑似泄漏各使用一个独立 Session。任务只给范围，Agent 自行查找规则、用 diagnostic_query 获取当前数据，再给出必要判断与可验证下一步。检查不能仅凭分类标签或任意文件读取通过。

In [ ]:
for kind in ('capacity', 'leak'):
    print(kind, lab.probe(kind, phase='before'))


## 3. 由 Curator 自己判断

下一格会调用 GLM。`prepare_review()` 建立快照、安装课程读取记录适配器，并限制工具与迭代次数；不指定合并目标，不替换原生整理提示。缺少凭据时隐藏询问。失败不是 no-op，保持原样也不是失败。


In [ ]:
lab.prepare_review()
try:
    returned = curator.run_curator_review(consolidate=True, synchronous=True)
    lab.observe_review(returned)
finally:
    lab._restore_observer()
lab.native['outcome']


## 4. 整理后用相同现场检查能力保留

再次运行同样两项任务，模型与快照保持相同，Session 和数据库独立。检查相关规则的完整条件、现场查询和引用、必要判断与下一步验证。规则可迁移、改名或合入共同 SKILL.md，验收不要求原文件名；也允许完整运行后的 no-op。

`comparison-capacity.json` 和 `comparison-leak.json` 比较前后原始输入、快照与检查结果。容量题应区分需求和归还；泄漏题不能在缺少释放代码时确认根因，专用下一步应覆盖正常、异常、提前返回及成功/失败请求验证。方法和计划通过不表示实际执行了代码验证，也不证明统计收益。

In [ ]:
for kind in ('capacity', 'leak'):
    print(kind, lab.probe(kind, phase='after'))


## 5. 单项恢复与全库回滚

`restore_skill(name)` 恢复某个归档技能；`curator_backup.rollback(snapshot_name)` 恢复整份快照。下面只在本轮教学副本中删除一个支持文件，检查失联，再恢复到整理前。即使模型选择 no-op，也能使用第一步归档的退役技能完成恢复练习。


In [ ]:
lab.recover()
result = lab.finish()
result


## 继续思考

哪些低频能力应保护？什么情况下不应合并？如何保留专用脚本及其引用？支持文件存在是否足以证明任务仍可完成？这些是设计判断，没有自动化唯一答案。本 Notebook 不验证周期调度或生产目录采用。
